In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine
from sqlalchemy import text
import urllib
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Configurar cadena de conexión
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_NAME')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [ ]:
query = """
SELECT DISTINCT c.Cliente,
	c.Documento,
    c.Nombres,
    c.Apellidos,
    c.Email,
    c.Celular,
    c.TipoIdentificacion
FROM Ventas_Comerssia.DBO.View_Clientes c
WHERE NOT EXISTS (SELECT 1 FROM Ventas_Comerssia.dbo.Ventas_Unificadas v WHERE c.Cliente = v.Cliente)
"""

# Ejecutar y cargar en DataFrame
df_clientes = pd.read_sql(query, engine)

df_clientes = df_clientes.drop_duplicates(subset='Cliente', keep='first')

# Obtener los CLICodigos ya existentes en la tabla SQL
# clientes_existentes = pd.read_sql("SELECT Cliente FROM View_Contacto_Clientes", con=engine)

In [4]:
#Limpiar Celular
def limpiar_celular(cel):
    if pd.isna(cel):
        return ""
    cel = re.sub(r'\D', '', str(cel))  # Elimina todo lo que no sea dígito
    if cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df_clientes['Celular'] = df_clientes['Celular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df_clientes['CelularValido'] = df_clientes['Celular'].apply(es_celular_valido)

#Limpiar y validar email
df_clientes['Email'] = df_clientes['Email'].apply(lambda x: str(x).strip())

def es_email_valido(email):
    email = str(email).strip()

    if pd.isna(email):
        return False

    email = str(email).strip()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["NEGAD", "PENDI", "negad","pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df_clientes['EmailValido'] = df_clientes['Email'].apply(es_email_valido)

In [5]:
ruta_lista_negra = r"C:\Users\Asus\OneDrive - PROVENZAL S.A.S\Escritorio\LISTA NEGRA.xlsx"
ruta_rne = r"C:\Users\Asus\OneDrive - PROVENZAL S.A.S\Escritorio\RNE Celular.xlsx"

df_blacklist = pd.read_excel(ruta_lista_negra)
df_rne = pd.read_excel(ruta_rne)

# -------------------
# LISTA NEGRA
# -------------------
df_blacklist['ID CLIENTE'] = df_blacklist['ID CLIENTE'].astype(str).str.strip()
df_blacklist['E_MAIL'] = df_blacklist['E_MAIL'].astype(str).str.strip().str.lower()

blacklist_clientes = set(df_blacklist['ID CLIENTE'].dropna())
blacklist_emails = set(df_blacklist['E_MAIL'].dropna())

# -------------------
# RNE
# -------------------
df_rne['numero'] = df_rne['numero'].astype(str).str.strip()
rne_celulares = set(df_rne['numero'].dropna())

# -------------------
# NORMALIZAR CLIENTES
# -------------------
df_clientes['Cliente'] = df_clientes['Cliente'].astype(str).str.strip()
df_clientes['Email'] = df_clientes['Email'].astype(str).str.strip().str.lower()
df_clientes['Celular'] = df_clientes['Celular'].astype(str).str.strip()

# -------------------
# VALIDACIONES
# -------------------
df_clientes['Cliente_Blacklist'] = df_clientes['Cliente'].isin(blacklist_clientes)
df_clientes['Email_Blacklist'] = df_clientes['Email'].isin(blacklist_emails)
df_clientes['Celular_RNE'] = df_clientes['Celular'].isin(rne_celulares)

In [6]:
# # Detectar duplicados
# cel_duplicados = df_clientes['Celular'].duplicated(keep=False)
# email_duplicados = df_clientes['Email'].duplicated(keep=False)

# # Marcar celulares duplicados como no válidos
# df_clientes.loc[cel_duplicados, 'CelularValido'] = False

# # Marcar correos duplicados como no válidos
# df_clientes.loc[email_duplicados, 'EmailValido'] = False

In [7]:
# Marcar celulares que están en RNE como no válidos
df_clientes['CelularValido'] = (
    df_clientes['CelularValido'] & 
    ~df_clientes['Celular_RNE']
)

# Marcar clientes que están en la blacklist como no válidos
df_clientes['EmailValido'] = (
    df_clientes['EmailValido'] & 
    ~df_clientes['Email_Blacklist']
)

#Agregar columna contacto
def determinar_contacto(row):

    # Bloqueo total por cliente en lista negra
    if row['Cliente_Blacklist']:
        return "Falso"

    celular_ok = row['CelularValido']
    email_ok = row['EmailValido']

    if celular_ok and email_ok:
        return "Cel + Email"
    elif celular_ok:
        return "Cel"
    elif email_ok:
        return "Email"
    else:
        return "Falso"

df_clientes['Contacto'] = df_clientes.apply(determinar_contacto, axis=1)

columnas_auxiliares = [
    'Cliente_Blacklist',
    'Email_Blacklist',
    'Celular_RNE'
]

df_exportar = df_clientes.drop(columns=columnas_auxiliares)

In [8]:
# # # Filtrar para dejar solo nuevos clientes
# df_nuevos = df_clientes[~df_clientes["Cliente"].isin(clientes_existentes["Cliente"])]

In [9]:
# # Insertar solo los nuevos en SQL
# if not df_nuevos.empty:
#     df_nuevos.to_sql("View_Contacto_Clientes", con=engine, if_exists="append", index=False)
#     print(f"{len(df_nuevos)} nuevos clientes insertados.")
# else:
#     print("No hay nuevos clientes por insertar.")

In [10]:
# Exportar todos los registros con validaciones incluidas
# df_clientes.to_excel("clientes_con_validacion.xlsx", index=False)

In [11]:
df_exportar.to_sql("Clientes", engine, if_exists="replace", index=False)

24